# GEO · E2 — relations as rotations (QuatE on the dictionary graph)

**What this does:** trains a from-scratch **QuatE** knowledge-graph embedder (entities as quaternion vectors, relations as **Hamilton-product rotations** — the same operation the Oracle's composition engine uses) on the dictionary's 3,033 concepts × 5,181 relations, then asks three pre-registered questions:

1. **LINK-LEARNABLE** — is the hand-built graph regular enough to predict held-out relations? (filtered MRR / Hits@k; same 80/20 split as E1/E1.5)
2. **OPERATOR-ORDERING** — do the *learned* rotation angles rediscover the contract's ordering (synonym ≈ identity < affinity < complement)? *Honesty note: QuatE's rotations live in its own latent space, so the test is the ordering, never absolute 90°.*
3. **TOPOLOGY→METRIC** — QuatE never sees the grounded angles, only edge types. Do its entity angles nonetheless correlate with the grounded metric on held-out pairs? (If yes: the graph topology alone carries the metric.)

**Deliverable:** ranked **candidate-relation mining** — top proposed new complement / synonym / opposition pairs not currently in the graph, each with its grounded 7D angle attached for review (candidates are *proposals*; nothing enters the dictionary without vetting). Known gap pairs (creation/destruction, peace/violent…) are rank-checked explicitly.

**How to run:** Runtime ▸ Change runtime type ▸ **T4 GPU** ▸ Save · Runtime ▸ **Run all**. ≈15–25 min. Output: `geo_e2_results.zip` (browser download at the end; optional Drive cell last).

*Staged 2026-08-20 · ladder rung E2 (pass doc §6) · lineage: QuatE, Zhang et al. 2019, arXiv:1904.10281*

In [ ]:
# ── Setup: GPU check, repo clone (no installs needed — pure torch) ────────────
import subprocess
from pathlib import Path
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NONE DETECTED')
import torch
assert torch.cuda.is_available(), (
    'No GPU. Colab menu: Runtime > Change runtime type > Hardware accelerator: T4 GPU, then Run all again.')

REPO = Path('/content/agi-semantic-core')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1',
        'https://github.com/QAv2/agi-semantic-core.git', str(REPO)], check=True)
DB = REPO/'db'/'semantic.db'
assert DB.exists(), 'semantic.db missing from clone'
RESULTS = Path('/content/geo2_results'); RESULTS.mkdir(exist_ok=True)
import sqlite3
print('concepts:', sqlite3.connect(DB).execute('SELECT COUNT(*) FROM concepts').fetchone()[0], '(expected 3033)')
SEED = 42

In [ ]:
# ── Dictionary extraction — IDENTICAL to E1/E1.5 (same seed → same splits) ────
import numpy as np, sqlite3
from collections import Counter
rng = np.random.default_rng(SEED)

con = sqlite3.connect(DB); con.row_factory = sqlite3.Row
DIMS = ['x','y','z','e','f','g','h','fx','fy','fz','fe','ff','fg','fh']
rows = con.execute(f"SELECT id,name,description,{','.join(DIMS)} FROM concepts ORDER BY id").fetchall()
names = [r['name'] for r in rows]
texts = [f"{r['name']}: {r['description']}" for r in rows]
Y14 = np.array([[r[d] for d in DIMS] for r in rows], dtype=np.float64)
Y7 = Y14[:, :7]
id2idx = {r['id']: i for i, r in enumerate(rows)}

pairs = []
for r in con.execute("SELECT concept1_id c1, concept2_id c2, rel_type, angle_4d, angle_8d FROM relations"):
    if r['c1'] not in id2idx or r['c2'] not in id2idx: continue
    ta = r['angle_8d'] if r['angle_8d'] else r['angle_4d']
    if not ta: continue
    pairs.append((id2idx[r['c1']], id2idx[r['c2']], r['rel_type'], float(ta)))
print(f'{len(pairs)} usable relations —', dict(Counter(p[2] for p in pairs)))

by_type = {}
for p in pairs: by_type.setdefault(p[2], []).append(p)
train_rel, test_rel = [], []
for t, ps in sorted(by_type.items()):
    idx = rng.permutation(len(ps)); cut = int(0.8*len(ps))
    train_rel += [ps[i] for i in idx[:cut]]; test_rel += [ps[i] for i in idx[cut:]]
print(f'relation split: {len(train_rel)} train / {len(test_rel)} held out (same as E1/E1.5)')

def angle7(i, j):
    a, b = Y7[i], Y7[j]
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-9 or nb < 1e-9: return None
    return float(np.degrees(np.arccos(np.clip(a@b/(na*nb), -1, 1))))
related = {(min(p[0],p[1]), max(p[0],p[1])) for p in pairs}
rand_pairs, seen = [], set()
while len(rand_pairs) < 12000:
    i, j = (int(v) for v in rng.integers(0, len(names), 2))
    key = (min(i,j), max(i,j))
    if i == j or key in related or key in seen: continue
    a = angle7(i, j)
    if a is None: continue
    seen.add(key); rand_pairs.append((i, j, 'random', a))
rand_train, rand_test = rand_pairs[:10000], rand_pairs[10000:]
print(f'random-pair channel: {len(rand_train)} train / {len(rand_test)} held out')

In [ ]:
# ── QuatE: entities = quaternion vectors, relations = Hamilton rotations ─────
import torch, torch.nn as nn, numpy as np, time
torch.manual_seed(SEED); np.random.seed(SEED)
dev = 'cuda'
REL_TYPES = sorted({p[2] for p in pairs})
r2i = {r: i for i, r in enumerate(REL_TYPES)}
N, M, K = len(names), len(REL_TYPES), 24          # K quaternion components = 96 real dims
print('relation vocabulary:', REL_TYPES)

def edges_of(plist):
    e = [(p[0], r2i[p[2]], p[1]) for p in plist]
    e += [(p[1], r2i[p[2]], p[0]) for p in plist]  # all rel types treated symmetric
    return torch.tensor(e, dtype=torch.long, device=dev)
train_e, test_e = edges_of(train_rel), edges_of(test_rel)

# filter map for ranking eval: (h, r) -> known true tails across ALL edges
from collections import defaultdict
known_tails = defaultdict(list)
for p in pairs:
    known_tails[(p[0], r2i[p[2]])].append(p[1])
    known_tails[(p[1], r2i[p[2]])].append(p[0])

class QuatE(nn.Module):
    def __init__(self, n, m, k):
        super().__init__()
        self.E = nn.Parameter(torch.randn(n, k, 4) * 0.1)
        self.R = nn.Parameter(torch.randn(m, k, 4) * 0.1)
    @staticmethod
    def hamilton(q, p):
        a1,b1,c1,d1 = q.unbind(-1); a2,b2,c2,d2 = p.unbind(-1)
        return torch.stack([a1*a2 - b1*b2 - c1*c2 - d1*d2,
                            a1*b2 + b1*a2 + c1*d2 - d1*c2,
                            a1*c2 - b1*d2 + c1*a2 + d1*b2,
                            a1*d2 + b1*c2 - c1*b2 + d1*a2], -1)
    def rotate(self, h_idx, r_idx):
        r = self.R[r_idx]
        r = r / r.norm(dim=-1, keepdim=True).clamp_min(1e-9)   # unit quaternion = pure rotation
        return self.hamilton(self.E[h_idx], r)
    def score_all(self, h_idx, r_idx):
        hr = self.rotate(h_idx, r_idx).reshape(len(h_idx), -1)
        return hr @ self.E.reshape(N, -1).T                    # [B, N] — 1-vs-all scores

model = QuatE(N, M, K).to(dev)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)
lossf = nn.CrossEntropyLoss()

def eval_mrr(edges):
    model.eval()
    ranks = []
    with torch.no_grad():
        for s in range(0, len(edges), 512):
            b = edges[s:s+512]
            sc = model.score_all(b[:,0], b[:,1])
            for row in range(len(b)):
                h, r, t = (int(v) for v in b[row])
                gold = sc[row, t].item()
                masked = sc[row].clone()
                for kt in known_tails[(h, r)]: masked[kt] = -1e9
                rank = int((masked > gold).sum().item()) + 1
                ranks.append((r, rank))
    model.train()
    return ranks

EPOCHS, B = 400, 1024
t0 = time.time()
for ep in range(1, EPOCHS+1):
    perm = torch.randperm(len(train_e), device=dev)
    tot = 0.0
    for s in range(0, len(train_e), B):
        b = train_e[perm[s:s+B]]
        loss = lossf(model.score_all(b[:,0], b[:,1]), b[:,2])
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(b)
    if ep % 100 == 0 or ep == 1:
        ranks = eval_mrr(test_e)
        mrr = float(np.mean([1.0/r for _, r in ranks]))
        print(f'epoch {ep:3d}  loss {tot/len(train_e):.4f}  test MRR {mrr:.3f}  ({time.time()-t0:.0f}s)')
print('training done')

In [ ]:
# ── Eval A: filtered link prediction on held-out relations ────────────────────
import numpy as np, pandas as pd, json
ranks = eval_mrr(test_e)
rowsA = []
for label, sel in [('OVERALL', ranks)] + [(t, [(r, k) for r, k in ranks if r == r2i[t]]) for t in REL_TYPES]:
    if not sel: continue
    ks = np.array([k for _, k in sel], dtype=float)
    rowsA.append({'relation': label, 'n': len(ks), 'MRR': float((1/ks).mean()),
                  'Hits@1': float((ks <= 1).mean()), 'Hits@3': float((ks <= 3).mean()),
                  'Hits@10': float((ks <= 10).mean())})
dfA = pd.DataFrame(rowsA).set_index('relation')
print(dfA.round(3))
dfA.to_csv(RESULTS/'e2_linkpred.csv')

In [ ]:
# ── Eval B: operator geometry + topology→metric ───────────────────────────────
import numpy as np, pandas as pd, json, torch
from scipy.stats import spearmanr, pearsonr

with torch.no_grad():
    Rn = model.R / model.R.norm(dim=-1, keepdim=True).clamp_min(1e-9)   # [M,K,4]
    theta = torch.rad2deg(2*torch.arccos(Rn[..., 0].abs().clamp(0, 1))) # rotation angle per component
rowsB = []
for t in REL_TYPES:
    th = theta[r2i[t]].cpu().numpy()
    rowsB.append({'relation': t, 'rot_mean': float(th.mean()),
                  'rot_median': float(np.median(th)), 'rot_std': float(th.std())})
dfB = pd.DataFrame(rowsB).sort_values('rot_mean').set_index('relation')
print(dfB.round(1))
dfB.to_csv(RESULTS/'e2_rotations.csv')

with torch.no_grad():
    Eflat = model.E.reshape(N, -1)
    Eflat = (Eflat / Eflat.norm(dim=1, keepdim=True)).cpu().numpy()
def qangle(i, j):
    return float(np.degrees(np.arccos(np.clip(Eflat[i] @ Eflat[j], -1, 1))))
tm_pairs = test_rel + rand_test
true_a = np.array([p[3] for p in tm_pairs])
pred_a = np.array([qangle(p[0], p[1]) for p in tm_pairs])
sp = float(spearmanr(true_a, pred_a)[0]); pe = float(pearsonr(true_a, pred_a)[0])
print(f'topology→metric (held-out relations + random pairs, n={len(tm_pairs)}): '
      f'Spearman {sp:.3f}, Pearson {pe:.3f}')
json.dump({'spearman': sp, 'pearson': pe, 'n': len(tm_pairs)},
          open(RESULTS/'e2_topometric.json','w'), indent=1)

In [ ]:
# ── Eval C: candidate-relation mining (proposals, grounded angle attached) ────
import numpy as np, pandas as pd, torch
existing = set()
for p in pairs:
    existing.add((min(p[0],p[1]), max(p[0],p[1])))

MINE = ['complement', 'synonym', 'opposition']
gap_probe = ['CREATION','CREATE','DESTRUCTION','DESTROY','PEACE','VIOLENT','VIOLENCE',
             'SILENCE','NOISE','VICE']
present = {n: i for i, n in enumerate(names)}
print('gap-probe concepts present:', [g for g in gap_probe if g in present])

for rel in MINE:
    ri = torch.tensor([r2i[rel]]*512, device=dev)
    S = np.zeros((N, N), dtype=np.float32)
    with torch.no_grad():
        for s in range(0, N, 512):
            h = torch.arange(s, min(s+512, N), device=dev)
            S[s:s+512] = model.score_all(h, ri[:len(h)]).cpu().numpy()
    Ssym = (S + S.T) / 2
    iu = np.triu_indices(N, k=1)
    order = np.argsort(-Ssym[iu])
    rows_out, count = [], 0
    for oi in order:
        i, j = int(iu[0][oi]), int(iu[1][oi])
        if (i, j) in existing: continue
        ga = angle7(i, j)
        rows_out.append({'concept1': names[i], 'concept2': names[j],
                         'score': float(Ssym[i, j]),
                         'grounded_angle_7d': None if ga is None else round(ga, 1),
                         'in_complement_band': (ga is not None and 60 <= ga <= 120)})
        count += 1
        if count >= 150: break
    df = pd.DataFrame(rows_out)
    df.to_csv(RESULTS/f'e2_candidates_{rel}.csv', index=False)
    print(f'\n=== top 15 proposed NEW {rel} pairs (of 150 saved) ===')
    print(df.head(15).to_string(index=False))
    # rank-check the named gap pairs under this relation
    for a in ['CREATION','CREATE']:
        for b in ['DESTRUCTION','DESTROY']:
            if a in present and b in present:
                i, j = present[a], present[b]
                sc = Ssym[i, j]
                pct = float((Ssym[iu] < sc).mean() * 100)
                print(f'  gap check {a}–{b} [{rel}]: score {sc:.2f} (beats {pct:.1f}% of all pairs)')
    if 'PEACE' in present and 'VIOLENT' in present:
        i, j = present['PEACE'], present['VIOLENT']
        sc = Ssym[i, j]; pct = float((Ssym[iu] < sc).mean() * 100)
        print(f'  gap check PEACE–VIOLENT [{rel}]: score {sc:.2f} (beats {pct:.1f}% of all pairs)')
np.savez_compressed(RESULTS/'e2_embeddings.npz',
                    E=model.E.detach().cpu().numpy(), R=model.R.detach().cpu().numpy(),
                    names=np.array(names), rel_types=np.array(REL_TYPES))

In [ ]:
# ── Pre-registered verdict ────────────────────────────────────────────────────
import json, pandas as pd, numpy as np
dfA = pd.read_csv(RESULTS/'e2_linkpred.csv').set_index('relation')
dfB = pd.read_csv(RESULTS/'e2_rotations.csv').set_index('relation')
tm = json.load(open(RESULTS/'e2_topometric.json'))
overall = dfA.loc['OVERALL']

q1 = bool(overall.MRR >= 0.15 and overall['Hits@10'] >= 0.30)
rot = dfB.rot_mean
q2 = bool(rot.get('synonym', 1e9) < rot.get('affinity', -1) < rot.get('complement', -1))
q3 = bool(abs(tm['spearman']) >= 0.30)

print('Q1 LINK-LEARNABLE   (MRR>=0.15 & Hits@10>=0.30):', 'PASS' if q1 else 'FAIL',
      f"  [MRR {overall.MRR:.3f}, Hits@10 {overall['Hits@10']:.3f}]")
print('Q2 OPERATOR-ORDERING (synonym < affinity < complement):', 'PASS' if q2 else 'FAIL',
      '  [' + ', '.join(f'{t} {rot[t]:.0f}deg' for t in rot.index) + ']')
print('Q3 TOPOLOGY->METRIC  (|Spearman|>=0.30):', 'PASS' if q3 else 'FAIL',
      f"  [Spearman {tm['spearman']:.3f}]")
verdict = {'Q1_link_learnable': q1, 'Q2_operator_ordering': q2, 'Q3_topology_metric': q3,
           'overall_mrr': float(overall.MRR), 'hits10': float(overall['Hits@10']),
           'rotation_means': {t: float(rot[t]) for t in rot.index},
           'topometric_spearman': tm['spearman']}
json.dump(verdict, open(RESULTS/'e2_verdict.json','w'), indent=1)
print()
print('Interpretation notes: Q2 tests the ORDERING only — QuatE angles live in its own')
print('latent space, so absolute 90 is not the test. Q3 positive means the graph topology')
print('alone (edge types, no angles) reconstructs part of the grounded metric. Mining CSVs')
print('are PROPOSALS for review — nothing enters the dictionary without vetting.')

In [ ]:
# ── Package results (+ browser download) ──────────────────────────────────────
import shutil
from pathlib import Path
PKG = Path('/content/geo2_pkg'); shutil.rmtree(PKG, ignore_errors=True); PKG.mkdir()
for f in ['e2_linkpred.csv','e2_rotations.csv','e2_topometric.json','e2_verdict.json',
          'e2_candidates_complement.csv','e2_candidates_synonym.csv',
          'e2_candidates_opposition.csv','e2_embeddings.npz']:
    p = RESULTS/f
    if p.exists(): shutil.copy(p, PKG/f)
zip_path = shutil.make_archive('/content/geo_e2_results', 'zip', PKG)
print('packaged:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print('manual download: use the Files sidebar ->', zip_path)

In [ ]:
# ── OPTIONAL: also copy results to your Google Drive ─────────────────────────
SAVE_TO_DRIVE = False   # flip to True, then run this cell
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    dest = '/content/drive/MyDrive/geo_e2_results.zip'
    shutil.copy('/content/geo_e2_results.zip', dest)
    print('saved to Drive:', dest)